In [1]:
import pandas as pd
posts = pd.read_csv("/content/posts.csv")
comments = pd.read_csv("/content/reddit_comments_clean.csv")



In [2]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)


RENAMING COLUMNS

In [3]:
posts = posts.rename(columns={"id": "post_id", "text": "post_text"})
comments = comments.rename(columns={"text": "comment_text"})

FILTER OUT EMPTY TEXTS

In [4]:
posts["post_text"].isna().sum(), comments["comment_text"].isna().sum()
comments = comments.dropna(subset=["comment_text"])
posts["post_text"] = posts["post_text"].fillna("[EXTERNAL_LINK_OR_NO_TEXT]")


sort everything by time

In [5]:
posts = posts.sort_values("timestamp")
comments = comments.sort_values("timestamp")


grouping commnets by post


In [6]:
grouped_comments = comments.groupby("post_id")


BUILDING NARRATIVE UNITS EXPLICITLY

In [7]:
narrative_units = []

for _, post in posts.iterrows():
    pid = post["post_id"]

    post_text = post["post_text"]
    post_time = post["timestamp"]

    if pid in grouped_comments.groups:
        comment_texts = grouped_comments.get_group(pid)["comment_text"].tolist()
    else:
        comment_texts = []

    narrative_units.append({
        "post_id": pid,
        "post_text": post_text,
        "post_timestamp": post_time,
        "comments": comment_texts,
        "num_comments": len(comment_texts)
    })


CONVERTING NARRATIVE UNITS TO DATAFRAME


In [8]:
narratives_df = pd.DataFrame(narrative_units)
narratives_df.head()



post_id  \
0  12vyubs   
1  132oa6r   
2  134393k   
3  1345vai   
4  1347y8a   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    post_text  \
0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            

importing transformer for sentence embedding


In [9]:
!pip install sentence-transformers nltk


note: here nltk is a natural language toolkit , will used it for sentence splitting.

punkt is a sentence tokenizer(i.e tool that splits text into meaningful units)

In [ ]:
from sentence_transformers import SentenceTransformer
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')


LOAD AN EMBEEDDIG MODEL

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")


EXTRACTING SENTENCES FROM NARRATIVE UNITS

sentence split function


In [ ]:
from nltk.tokenize import sent_tokenize

def extract_sentences(text):
    if not isinstance(text, str):
        return []
    return sent_tokenize(text)


build a sentence level dataset

In [ ]:
sentence_records = []

# Create lookup for post timestamps
post_time_lookup = posts.set_index("post_id")["timestamp"].to_dict()

for _, row in narratives_df.iterrows():
    post_id = row["post_id"]

    # --- Post sentences ---
    post_sentences = extract_sentences(row["post_text"])
    post_timestamp = post_time_lookup.get(post_id)

    for s in post_sentences:
        sentence_records.append({
            "post_id": post_id,
            "source": "post",
            "sentence": s,
            "timestamp": post_timestamp
        })

    # --- Comment sentences ---
    post_comments = comments[comments["post_id"] == post_id]

    for _, c in post_comments.iterrows():
        comment_sentences = extract_sentences(c["comment_text"])
        comment_timestamp = c["timestamp"]

        for s in comment_sentences:
            sentence_records.append({
                "post_id": post_id,
                "source": "comment",
                "sentence": s,
                "timestamp": comment_timestamp
            })


In [ ]:
sentences_df = pd.DataFrame(sentence_records)
sentences_df.head()


(issue fixing : sentence level duplication)

In [ ]:
sentences_df = sentences_df.drop_duplicates(
    subset=["sentence", "timestamp", "post_id", "source"]
).reset_index(drop=True)


GENERATING EMBEDDINGS


In [ ]:
embeddings = model.encode(
    sentences_df["sentence"].tolist(),
    show_progress_bar=True
)
sentences_df["embedding"] = list(embeddings)


PHASE 4: NARRATIVE CLUSTERING & EMERGENCE





INSTALLING CLUSTERING ALGO
1. HDBSCAN
2. UMAP



In [ ]:
!pip install hdbscan umap-learn


preparing embeddings for clusterings

In [ ]:

import numpy as np
import hdbscan


prepare time + embeddings

In [ ]:
# Ensure timestamp is datetime
sentences_df["timestamp"] = pd.to_datetime(sentences_df["timestamp"])

# Convert embeddings column to NumPy array (once)
X_all = np.vstack(sentences_df["embedding"].values)


dimensionality reduction ONCE

In [ ]:
import umap

reducer = umap.UMAP(
    n_neighbors=15,
    n_components=50,   # ↓ from 384
    metric="cosine",
    random_state=42
)

X_reduced = reducer.fit_transform(X_all)


attach reduced embeddings back


In [ ]:
sentences_df["embedding_reduced"] = list(X_reduced)


72-hour sliding windows

In [ ]:
from datetime import timedelta
sentences_df["timestamp"] = pd.to_datetime(sentences_df["timestamp"])

WINDOW_SIZE = timedelta(hours=72)
STEP_SIZE = timedelta(hours=24)

start_time = sentences_df["timestamp"].min()
end_time = sentences_df["timestamp"].max()


In [ ]:
window_results = []

current_start = start_time

while current_start < end_time:
    current_end = current_start + WINDOW_SIZE

    window_df = sentences_df[
        (sentences_df["timestamp"] >= current_start) &
        (sentences_df["timestamp"] < current_end)
    ]

    if len(window_df) < 50:
        current_start += STEP_SIZE
        continue

    X_window = np.vstack(window_df["embedding_reduced"].values)

    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=30,
        metric="euclidean"
    )

    labels = clusterer.fit_predict(X_window)

    window_df = window_df.copy()
    window_df["cluster"] = labels
    window_df["window_start"] = current_start

    window_results.append(window_df)

    current_start += STEP_SIZE


reassemble clustered data

In [ ]:
clustered_sentences_df = pd.concat(window_results, ignore_index=True)
clustered_sentences_df.head()


# **phase 4.2 :narrative continuity across windows**

compute centroids for each window-cluster

What this does (conceptually):

Takes each cluster in each window,
Computes its “average meaning”,
Stores it with window + size

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

cluster_centroids = []

for (window_start, cluster_id), group in clustered_sentences_df.groupby(
    ["window_start", "cluster"]
):
    if cluster_id == -1:
        continue  # ignore noise

    embeddings = np.vstack(group["embedding_reduced"].values)
    centroid = embeddings.mean(axis=0)

    cluster_centroids.append({
        "window_start": window_start,
        "cluster": cluster_id,
        "centroid": centroid,
        "size": len(group)
    })

centroids_df = pd.DataFrame(cluster_centroids)


link clusters across adjacent windows

What this does :
First window: everything is “new”
Each next window:
Compare clusters to previous window
If meaning is similar → same narrative
If not → new narrative

In [ ]:
SIMILARITY_THRESHOLD = 0.65  # conservative, journalist-safe

narrative_id = 0
narrative_map = {}

centroids_df = centroids_df.sort_values("window_start")

previous_window_centroids = None

for window_start, window_group in centroids_df.groupby("window_start"):
    if previous_window_centroids is None:
        # First window → every cluster starts a new narrative
        for _, row in window_group.iterrows():
            narrative_map[(window_start, row["cluster"])] = narrative_id
            narrative_id += 1
        previous_window_centroids = window_group
        continue

    for _, row in window_group.iterrows():
        current_centroid = row["centroid"]

        sims = cosine_similarity(
            [current_centroid],
            np.vstack(previous_window_centroids["centroid"].values)
        )[0]

        max_sim = sims.max()

        if max_sim >= SIMILARITY_THRESHOLD:
            matched_idx = sims.argmax()
            matched_row = previous_window_centroids.iloc[matched_idx]
            matched_narrative = narrative_map[
                (matched_row["window_start"], matched_row["cluster"])
            ]
            narrative_map[(window_start, row["cluster"])] = matched_narrative
        else:
            narrative_map[(window_start, row["cluster"])] = narrative_id
            narrative_id += 1

    previous_window_centroids = window_group


attach persistent narrative IDs back to sentences

In [ ]:
def get_narrative_id(row):
    return narrative_map.get((row["window_start"], row["cluster"]))

clustered_sentences_df["narrative_id"] = clustered_sentences_df.apply(
    get_narrative_id,
    axis=1
)


# **phase 5:narrative justification & evidence layer**

Aggregate Narrative Statistics

In [ ]:
narrative_stats = (
    clustered_sentences_df
    .drop_duplicates(subset=["sentence", "timestamp", "post_id"])
    .groupby("narrative_id")
    .agg(
        total_sentences=("sentence", "count"),
        unique_posts=("post_id", "nunique"),
        windows_present=("window_start", "nunique"),
        first_seen=("timestamp", "min"),
        last_seen=("timestamp", "max")
    )
    .reset_index()
)



Qualification Rules (Transparent Filtering)

In [ ]:
qualified_narratives = narrative_stats[
    (narrative_stats["total_sentences"] >= 15) &
    (narrative_stats["windows_present"] >= 2) &
    (narrative_stats["unique_posts"] >= 2)
]


Select Representative Evidence

1.   Pick sentences closest to the narrative centroid
2.   These are the most “typical” expressions



In [ ]:
evidence_records = []

for _, row in qualified_narratives.iterrows():
    nid = row["narrative_id"]

    narrative_sentences = clustered_sentences_df[
        clustered_sentences_df["narrative_id"] == nid
    ]

    centroid = np.vstack(
        narrative_sentences["embedding_reduced"].values
    ).mean(axis=0)

    sims = cosine_similarity(
        np.vstack(narrative_sentences["embedding_reduced"].values),
        [centroid]
    ).flatten()

    narrative_sentences = narrative_sentences.copy()
    narrative_sentences["similarity"] = sims

    top_evidence = (
    narrative_sentences
    .drop_duplicates(subset=["sentence", "post_id", "timestamp"])
    .sort_values("similarity", ascending=False)
    .head(10)
)


    for _, ev in top_evidence.iterrows():
        evidence_records.append({
            "narrative_id": nid,
            "sentence": ev["sentence"],
            "timestamp": ev["timestamp"],
            "post_id": ev["post_id"],
            "source": ev["source"]
        })

evidence_df = pd.DataFrame(evidence_records)


Model Auditing(part of phase 5)

Build a Clean Narrative Summary Table.

which will give us:

1.   How many narratives exist?

2.   Which ones dominate?

Which ones persist?













In [ ]:
narrative_summaries = qualified_narratives.copy()
narrative_summaries["duration_days"] = (
    narrative_summaries["last_seen"] - narrative_summaries["first_seen"]
).dt.days

narrative_summaries = narrative_summaries.sort_values(
    "total_sentences", ascending=False
)

narrative_summaries[
    [
        "narrative_id",
        "first_seen",
        "last_seen",
        "duration_days",
        "total_sentences",
        "unique_posts",
        "windows_present"
    ]
]


Inspect One Narrative Manually (Critical Sanity Check)

 inspection cell to see what will be displayed on dashboard (engine pov)

In [ ]:
NID = narrative_summaries.iloc[0]["narrative_id"]

print("=== NARRATIVE METADATA ===")
display(
    narrative_summaries[
        narrative_summaries["narrative_id"] == NID
    ]
)

print("=== EVIDENCE (what journalist will read) ===")
display(
    evidence_df[
        evidence_df["narrative_id"] == NID
    ][["sentence", "timestamp", "post_id", "source"]]
)


In [ ]:
narrative_stats[
    ["narrative_id", "total_sentences", "windows_present", "unique_posts"]
].sort_values(
    ["windows_present", "total_sentences"],
    ascending=False
)
